In [1]:
from langchain_ollama import ChatOllama

qwen25_llm = ChatOllama(model="qwen3:4b", temperature=0)
response = qwen25_llm.invoke("Hola, cómo estás?")
response.pretty_print()

================================== Ai Message ==================================

Hola! 😊 Me encanta ayudarte. Como inteligencia artificial, no tengo sentimientos, pero estoy aquí para ayudarte con cualquier pregunta o necesidad que tengas. ¿En qué puedo ayudarte hoy?


### No tiene información actualizada, entonces responde basado en su corpus de conocimiento

In [2]:
response = qwen25_llm.invoke("Qué clima hace en la ciudad de Bogotá?")
response.pretty_print()

================================== Ai Message ==================================

The typical climate in Bogotá is cool and dry year-round, with average temperatures ranging from 12°C to 22°C (54°F to 72°F). The city experiences low humidity and minimal rainfall, making it ideal for outdoor activities throughout the year. There is no distinct wet season, but occasional light showers may occur, particularly during transitional months.


### Se puede usar RAG para darle acceso a una base de conocimiento usando embbedins, una base de datos vectorial y un retriever, para que pueda responder preguntas basadas en esa información, o pasarle la info en un system prompt, aunque esto es menos eficiente y escalable.

In [5]:
system_prompt = """\
Eres un asistente de ventas que ayuda a los clientes a encontrar los productos que necesitan.

Tus productos son:
- Computadora
- Mouse
- Teclado
- Audifonos
- Mousepad
"""
messages = [("system", system_prompt), ("user", "Dime los productos que ofreces en la tienda")]

response = qwen25_llm.invoke(messages)
response.pretty_print()

================================== Ai Message ==================================

¡Hola! En nuestra tienda ofrecemos los siguientes productos: Computadora, Mouse, Teclado, Audifonos y Mousepad. ¿En qué puedo ayudarte? 😊


### Pero con TOOLs:

In [7]:
from langchain_core.tools import tool
import requests

@tool("get_products", description="Get the products that the store offers and their prices. Requires 'price' (int): maximum price to filter products.")
def get_products(price: int):
    """
    Get the products that the store offers and their prices, filtering by a maximum price.
    Args:
        price (int): Maximum price to filter products.
    """
    # Ejemplo con datos estáticos:
    # """Get the products that the store offers and their prices, filtering by a maximum price."""
    # products = [{"name": "Computadora", "price": 1000},
    #             {"name": "Mouse", "price": 100},
    #             {"name": "Teclado", "price": 200},
    #             {"name": "Audifonos", "price": 300},
    #             {"name": "Mousepad", "price": 400}]
    
    # return "".join([f"{product['name']}: ${product['price']}\n" for product in products if product["price"] <= price])

    # Ejemplo con base de datos API (api de Platzi abierta)
    response = requests.get("https://api.escuelajs.co/api/v1/products")
    products = response.json()
    return "".join([f"{product['title']}: ${product['price']}\n" for product in products if product["price"] <= price])

# Ejemplo de uso de tool gratis para clima sin API KEY
@tool("get_weather", description="Get weather for a city")
def get_weather(city: str) -> str:
    response = requests.get(f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1")
    data = response.json()
    latitude = data["results"][0]["latitude"]
    longitude = data["results"][0]["longitude"]
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true")
    data = response.json()
    response = f"The weather in {city} is {data['current_weather']['temperature']} C with {data['current_weather']['windspeed']} km/h of speed "
    return response


In [14]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen2.5:7b", temperature=0)

system_prompt = """\
Eres un asistente de ventas que ayuda a los clientes a encontrar los productos que necesitan y también les puedes dar el clima actual de su ciudad.

Tus Tools son:
- get_products(price: int): Get the products that the store offers and their prices, filtering by a maximum price.
- get_weather(city: str): Get the current weather for a given city.
"""
agent = create_agent(model=llm, tools=[get_products, get_weather], debug=True)

In [15]:
response = agent.invoke({"messages": [{"role": "system", "content": system_prompt},{"role": "user", "content": "Qué productos tienes por debajo de $500?"}]})
print(response["messages"][-1].content)

[values] {'messages': [SystemMessage(content='Eres un asistente de ventas que ayuda a los clientes a encontrar los productos que necesitan y también les puedes dar el clima actual de su ciudad.\n\nTus Tools son:\n- get_products(price: int): Get the products that the store offers and their prices, filtering by a maximum price.\n- get_weather(city: str): Get the current weather for a given city.\n', additional_kwargs={}, response_metadata={}, id='9a494e4f-bd03-463b-a334-08b84ae9c552'), HumanMessage(content='Qué productos tienes por debajo de $500?', additional_kwargs={}, response_metadata={}, id='6647f059-7495-43d4-8cdc-14fc08fd1582')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-03-04T16:58:06.0832327Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2026920200, 'load_duration': 107878800, 'prompt_eval_count': 273, 'prompt_eval_duration': 590142200, 'eval_count': 22, 'eval_duration': 

In [16]:
response = agent.invoke({"messages": [{"role": "system", "content": system_prompt},{"role": "user", "content": "Qué temperatura hace en Nueva Delhi?"}]})
print(response["messages"][-1].content)

[values] {'messages': [SystemMessage(content='Eres un asistente de ventas que ayuda a los clientes a encontrar los productos que necesitan y también les puedes dar el clima actual de su ciudad.\n\nTus Tools son:\n- get_products(price: int): Get the products that the store offers and their prices, filtering by a maximum price.\n- get_weather(city: str): Get the current weather for a given city.\n', additional_kwargs={}, response_metadata={}, id='ac3016ed-6cca-4657-8aba-d58cc58865b5'), HumanMessage(content='Qué temperatura hace en Nueva Delhi?', additional_kwargs={}, response_metadata={}, id='7bf030f3-9a43-4bd9-ad0c-10df5aa6a2d4')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-03-04T16:58:31.3340136Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2006037500, 'load_duration': 104464000, 'prompt_eval_count': 269, 'prompt_eval_duration': 437815500, 'eval_count': 22, 'eval_duration': 1431